In [ ]:
# ============================================================
# Cell 0: 环境自检（本章纯 CPU 即可，无需 GPU）
# ============================================================
# 本章只在长度 ≤ 8、维度 ≤ 64 的玩具张量上做矩阵乘法，全程 CPU 秒级跑完，
# 读 Qwen3 config 也只下载几 KB 的 json，所以不强制 GPU，只打印环境信息确认可用。
import sys, platform
import torch

print("Python:", sys.version.split()[0])
print("平台:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available(), "（本章用不到，CPU 即可）")

In [ ]:
%%capture
# ============================================================
# Cell 1: 安装依赖
# ============================================================
# %%capture 必须是 cell 第一行，把 pip 安装日志藏起来。
# torch:        张量运算 + F.scaled_dot_product_attention——Colab 默认已装且够新，
#               故意【不】加 -U：会话中途升级 torch 容易让内核半新半旧后续 import 报错。
# matplotlib:   画多头注意力权重热力图（不同头不同模式）。
# transformers: 第 7.7 节用 AutoConfig 读 Qwen3-8B 的 config（num_attention_heads 等）。
#               Qwen3 系列要求 transformers>=4.51，这里锁定版本下界。
!pip install -q -U matplotlib "transformers>=4.51"

In [ ]:
# ============================================================
# Cell 2: 复现第 6 章的单头 scaled dot-product attention
# ============================================================
# Q,K,V 形状 [..., L, d_k]，前置维 ... 可以是 batch B、也可以是 (B, H) 多头——
# 全靠 @ 的广播和 transpose(-2,-1) 只动最后两维，自动把前置维当 batch 并行。
import torch
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V, mask=None):
    """缩放点积注意力。Attention(Q,K,V) = softmax(QKᵀ / √d_k + mask) V
    参数:
      Q, K, V : [..., L, d_k]（这里取 d_v = d_k）
      mask    : 可选，可广播到 [..., L, L] 的布尔张量，True 表示该位置【要屏蔽】
    返回:
      out  : [..., L, d_k]   注意力输出
      attn : [..., L, L]     注意力权重（已 softmax，便于可视化）
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)      # ①QKᵀ ②缩放 -> [..., L, L]
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf"))  # 要屏蔽的位置打 -inf
    attn = F.softmax(scores, dim=-1)                      # ③逐行 softmax -> 权重
    out = attn @ V                                        # ④加权求和 -> [..., L, d_k]
    return out, attn

# 快速自测：带 (B, H) 两个前置维也能跑
Q = torch.randn(2, 4, 6, 16)   # [B=2, H=4, L=6, d_k=16]
out, attn = scaled_dot_product_attention(Q, Q, Q)
print("输入 [B,H,L,d_k]:", tuple(Q.shape))
print("输出 out :", tuple(out.shape), "  attn:", tuple(attn.shape))
print("→ B 和 H 都被当成 batch 维自动并行，最后两维 [L,d_k] 上做 attention")

In [ ]:
# ============================================================
# Cell 3: 从零实现 MultiHeadAttention（对应第 2、3 节）—— 重点看形状变换
# ============================================================
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    """标准多头注意力。X: [B, L, d_model] -> [B, L, d_model]（形状守恒）。"""
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
        self.d_model = d_model
        self.H = num_heads
        self.d_k = d_model // num_heads           # 每个头的维度
        # 四个投影：W_Q/W_K/W_V 把 d_model 投到 d_model（= 所有头合在一起），W_O 融合
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def split_heads(self, x):
        # [B, L, d_model] --reshape--> [B, L, H, d_k] --transpose--> [B, H, L, d_k]
        B, L, _ = x.shape
        x = x.reshape(B, L, self.H, self.d_k)     # 切头：把 d_model 劈成 H × d_k
        return x.transpose(1, 2)                   # 头维提前，最后两维变 [L, d_k]

    def merge_heads(self, x):
        # [B, H, L, d_k] --transpose--> [B, L, H, d_k] --reshape--> [B, L, d_model]
        B, H, L, d_k = x.shape
        x = x.transpose(1, 2).contiguous()         # 换回 [B, L, H, d_k]；transpose 后内存不连续，先 contiguous 重排成连续块（reshape 本身也会在不连续时自动 copy，这里显式写出更直观）
        return x.reshape(B, L, H * d_k)            # 拼接：H × d_k 合回 d_model

    def forward(self, X, causal=False, verbose=False):
        B, L, _ = X.shape
        Q = self.split_heads(self.W_Q(X))          # [B, H, L, d_k]
        K = self.split_heads(self.W_K(X))
        V = self.split_heads(self.W_V(X))
        if verbose:
            print(f"  输入 X            : {tuple(X.shape)}  [B, L, d_model]")
            print(f"  投影+切头后 Q     : {tuple(Q.shape)}  [B, H, L, d_k]")
        mask = None
        if causal:                                  # causal mask，[L,L] 自动广播到 [B,H,L,L]
            mask = torch.triu(torch.ones(L, L, dtype=torch.bool, device=X.device), diagonal=1)
        heads, attn = scaled_dot_product_attention(Q, K, V, mask=mask)  # [B,H,L,d_k], [B,H,L,L]
        merged = self.merge_heads(heads)            # 合头 -> [B, L, d_model]
        out = self.W_O(merged)                      # 输出投影，融合各头
        if verbose:
            print(f"  各头 attention 后 : {tuple(heads.shape)}  [B, H, L, d_k]")
            print(f"  合头后            : {tuple(merged.shape)}  [B, L, d_model]")
            print(f"  过 W_O 输出 out   : {tuple(out.shape)}  [B, L, d_model]（与输入同形）")
        return out, attn

torch.manual_seed(0)
B, L, d_model, H = 1, 5, 32, 4
mha = MultiHeadAttention(d_model, H)
X = torch.randn(B, L, d_model)
out, attn = mha(X, causal=True, verbose=True)
print("注意力权重 attn 形状:", tuple(attn.shape), " [B, H, L, L]——每个头一个 L×L 矩阵")

In [ ]:
# ============================================================
# Cell 4: 和 PyTorch 官方 nn.MultiheadAttention 对齐（对应第 2 节）
# ============================================================
# nn.MultiheadAttention 把 W_Q/W_K/W_V 打包成一个 in_proj_weight（按行堆叠），
# W_O 是 out_proj。我们把这些权重搬进自己的 MHA，相同输入应得到相同输出。
torch.manual_seed(0)
B, L, d_model, H = 2, 6, 32, 4
X = torch.randn(B, L, d_model)

ref = nn.MultiheadAttention(d_model, H, bias=False, batch_first=True)  # 官方实现
mine = MultiHeadAttention(d_model, H)

# 把官方的 in_proj_weight [3*d_model, d_model] 按 Q/K/V 三段拆开，灌进我们的三个 W
Wq, Wk, Wv = ref.in_proj_weight.chunk(3, dim=0)
with torch.no_grad():
    mine.W_Q.weight.copy_(Wq)
    mine.W_K.weight.copy_(Wk)
    mine.W_V.weight.copy_(Wv)
    mine.W_O.weight.copy_(ref.out_proj.weight)

out_mine, _ = mine(X, causal=False)
out_ref, _ = ref(X, X, X, need_weights=False)   # 自注意力：query=key=value=X

print("我们的输出形状:", tuple(out_mine.shape))
print("官方输出形状  :", tuple(out_ref.shape))
print("最大绝对误差  :", (out_mine - out_ref).abs().max().item())
assert torch.allclose(out_mine, out_ref, atol=1e-5), "与官方实现不一致！"
print("✅ 通过：手写多头注意力与 PyTorch 官方一致")

In [ ]:
# ============================================================
# Cell 5: 可视化——不同头学到不同的注意力模式（对应第 1.3、4.2 节）
# ============================================================
import matplotlib.pyplot as plt

torch.manual_seed(3)
B, L, d_model, H = 1, 8, 32, 4
X = torch.randn(B, L, d_model)
mha = MultiHeadAttention(d_model, H)
_, attn = mha(X, causal=True)        # [1, 4, 8, 8]，4 个头各一个 8×8 下三角注意力矩阵

fig, axes = plt.subplots(1, H, figsize=(4 * H, 3.6))
for h in range(H):
    ax = axes[h]
    im = ax.imshow(attn[0, h].detach().numpy(), cmap="viridis", vmin=0, vmax=1)
    ax.set_title(f"head {h + 1}")
    ax.set_xlabel("key j"); ax.set_ylabel("query i")
fig.colorbar(im, ax=axes, fraction=0.02, label="attention weight")
fig.suptitle("Different heads, different attention patterns (causal)")
plt.show()
print("→ 4 个头都是下三角（causal），但每个头把注意力压在不同的 key 上：")
print("  这是随机初始化、还没训练的头，但彼此的分布已经明显不一样——")
print("  这就是多头的价值：同一序列被多套独立的注意力分布同时审视。")

In [ ]:
# ============================================================
# Cell 6: 从零实现 GQA——query H 头，K/V 只 G 头，复制后参与计算（对应第 5 节）
# ============================================================
class GroupedQueryAttention(nn.Module):
    """GQA：num_heads 个 query 头，num_kv_heads 个 K/V 头（整除 num_heads）。
    num_kv_heads == num_heads -> MHA；num_kv_heads == 1 -> MQA。"""
    def __init__(self, d_model, num_heads, num_kv_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
        assert num_heads % num_kv_heads == 0, "query 头数须能被 KV 头数整除"
        self.H, self.G = num_heads, num_kv_heads
        self.d_k = d_model // num_heads
        self.rep = num_heads // num_kv_heads          # 每份 K/V 要复制几遍
        self.W_Q = nn.Linear(d_model, num_heads * self.d_k, bias=False)     # 投出 H 个 query 头
        self.W_K = nn.Linear(d_model, num_kv_heads * self.d_k, bias=False)  # 只投 G 个 K 头
        self.W_V = nn.Linear(d_model, num_kv_heads * self.d_k, bias=False)  # 只投 G 个 V 头
        self.W_O = nn.Linear(num_heads * self.d_k, d_model, bias=False)

    def forward(self, X, causal=False):
        B, L, _ = X.shape
        Q = self.W_Q(X).reshape(B, L, self.H, self.d_k).transpose(1, 2)  # [B, H, L, d_k]
        K = self.W_K(X).reshape(B, L, self.G, self.d_k).transpose(1, 2)  # [B, G, L, d_k]
        V = self.W_V(X).reshape(B, L, self.G, self.d_k).transpose(1, 2)  # [B, G, L, d_k]
        # 关键：把 G 个 KV 头沿头维各复制 rep 遍，扩成 H 个，之后就和 MHA 一样
        K = K.repeat_interleave(self.rep, dim=1)                          # [B, H, L, d_k]
        V = V.repeat_interleave(self.rep, dim=1)                          # [B, H, L, d_k]
        mask = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1) if causal else None
        heads, _ = scaled_dot_product_attention(Q, K, V, mask=mask)      # [B, H, L, d_k]
        out = heads.transpose(1, 2).contiguous().reshape(B, L, self.H * self.d_k)
        return self.W_O(out)

torch.manual_seed(0)
B, L, d_model, H = 1, 6, 64, 8
X = torch.randn(B, L, d_model)
for G in (8, 4, 1):                       # G=8 -> MHA, G=4 -> GQA, G=1 -> MQA
    gqa = GroupedQueryAttention(d_model, num_heads=H, num_kv_heads=G)
    out = gqa(X, causal=True)
    kv_params = sum(p.numel() for p in (gqa.W_K.weight, gqa.W_V.weight))
    name = {8: "MHA", 4: "GQA", 1: "MQA"}[G]
    print(f"{name:3s}  num_heads={H}  num_kv_heads={G}  "
          f"每个 KV 头被 {H // G} 个 query 头共享  "
          f"K+V 投影参数={kv_params:5d}  输出形状={tuple(out.shape)}")
print("→ query 头数恒为 8；KV 头数从 8(MHA)→4(GQA)→1(MQA)，K/V 投影参数随之减少，")
print("  推理时 KV cache 也按同比例缩小，而输出形状始终 [B, L, d_model] 不变。")

In [ ]:
# ============================================================
# Cell 7: 读真实 Qwen3-8B config，看 num_attention_heads vs num_key_value_heads
# ============================================================
# AutoConfig 只下载几 KB 的 config.json，不下载几个 GB 的权重，CPU 秒级完成。
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained("Qwen/Qwen3-8B")
H = cfg.num_attention_heads        # query 头数
G = cfg.num_key_value_heads        # K/V 头数（GQA 的组数）
d_model = cfg.hidden_size
head_dim = getattr(cfg, "head_dim", d_model // H)

print(f"hidden_size (d_model)      : {d_model}")
print(f"num_attention_heads (H)    : {H}")
print(f"num_key_value_heads (G)    : {G}")
print(f"head_dim (d_k)             : {head_dim}")
print(f"num_hidden_layers          : {cfg.num_hidden_layers}")
print("-" * 50)
kind = "MHA" if G == H else ("MQA" if G == 1 else "GQA")
print(f"→ 注意力类型: {kind}（H={H}, G={G}）")
print(f"  每个 KV 头被 {H // G} 个 query 头共享，KV cache 相对满血 MHA 缩小到 1/{H // G}")